In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv(r"C:\Projects\steam.csv")

In [4]:
df.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


In [5]:
# analyse data and drop unwanted columns

In [6]:
df.isnull().sum()

appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appid             27075 non-null  int64  
 1   name              27075 non-null  object 
 2   release_date      27075 non-null  object 
 3   english           27075 non-null  int64  
 4   developer         27074 non-null  object 
 5   publisher         27061 non-null  object 
 6   platforms         27075 non-null  object 
 7   required_age      27075 non-null  int64  
 8   categories        27075 non-null  object 
 9   genres            27075 non-null  object 
 10  steamspy_tags     27075 non-null  object 
 11  achievements      27075 non-null  int64  
 12  positive_ratings  27075 non-null  int64  
 13  negative_ratings  27075 non-null  int64  
 14  average_playtime  27075 non-null  int64  
 15  median_playtime   27075 non-null  int64  
 16  owners            27075 non-null  object

In [8]:
df.shape

(27075, 18)

## Data Cleaning

First i droped the unwanted columns

In [9]:
df = df.drop(columns= ["appid","developer","publisher","achievements"])

In [10]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year

In [11]:
df['owners'] = df['owners'].str.split('-').str[0]
df['owners'] = pd.to_numeric(df['owners'], errors='coerce')

In [12]:
df['rating_score'] = df['positive_ratings'] / (df['positive_ratings'] + df['negative_ratings'])
df['rating_score'] = df['rating_score'].fillna(0)

In [13]:
df.fillna('', inplace=True)

In [14]:
def clean_text(text):
    return str(text).lower().replace(';', ' ').replace(',', ' ')

In [15]:
cols = ['genres', 'categories', 'steamspy_tags', 'platforms']

for col in cols:
    df[col] = df[col].apply(clean_text)

In [16]:
df['combined'] = (
    df['genres'] + ' ' +
    df['categories'] + ' ' +
    df['steamspy_tags'] + ' ' +
    df['platforms']
)

In [17]:
df.columns

Index(['name', 'release_date', 'english', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'positive_ratings',
       'negative_ratings', 'average_playtime', 'median_playtime', 'owners',
       'price', 'year', 'rating_score', 'combined'],
      dtype='object')

In [18]:
df = df.drop([
    'release_date',
    'positive_ratings',
    'negative_ratings',
    'median_playtime',
    'genres',
    'categories',
    'steamspy_tags',
    'platforms'
], axis=1)

In [19]:
scaler = MinMaxScaler()

df[['rating_score', 'owners', 'average_playtime']] = scaler.fit_transform(
    df[['rating_score', 'owners', 'average_playtime']]
)

In [20]:
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
tfidf_matrix = tfidf.fit_transform(df['combined']).astype('float32')

In [21]:
def hybrid_recommend_fast(game_name, top_n=5):

    if game_name not in indices:
        return "Game not found"
    
    idx = indices[game_name]
    
    
    sim_scores = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()
    
    scores = []
    
    for i, sim in enumerate(sim_scores):
        if i == idx:
            continue
        
        score = (
            sim * 0.6 +
            df.iloc[i]['rating_score'] * 0.2 +
            df.iloc[i]['owners'] * 0.1 +
            df.iloc[i]['average_playtime'] * 0.1
        )
        
        scores.append((i, score))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_indices = [i[0] for i in scores[:top_n]]
    
    return df[['name', 'rating_score', 'price']].iloc[top_indices]

In [22]:
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [23]:
hybrid_recommend_fast("Counter-Strike", 5)

,name,rating_score,price
1,Team Fortress Classic,0.839787,3.99
3,Deathmatch Classic,0.826623,3.99
5,Ricochet,0.801278,3.99
7,Counter-Strike: Condition Zero,0.893871,7.19
6,Half-Life,0.961878,7.19
